In [1]:
# Clean conflicting stack (helps avoid torch/torchvision/triton mismatch)
!pip uninstall -y torch torchvision torchaudio triton bitsandbytes -q

# Install compatible CUDA stack
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0
!pip install -q --no-cache-dir \
  bitsandbytes==0.45.5 triton==3.2.0 \
  transformers==4.46.3 accelerate==0.34.2 \
  sentence-transformers==3.0.1 faiss-cpu==1.8.0.post1 \
  pandas==2.2.2 numpy==1.26.4 scipy==1.11.4 tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 303.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 417.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 224.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 254.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 176.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 240.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 283.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 180.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 277.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 255.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 240.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 257.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)

In [3]:
import json
import time
import random
from pathlib import Path
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [4]:
# ---- Model ----
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# ---- Generation settings ----
MAX_NEW_TOKENS = 80
TEMPERATURE = 0.0
TOP_K_RAG = 6
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SLEEP_SEC = 0.0
SEED = 42

# ---- DP prompt control (None = full book) ----
DP_MAX_BOOK_CHARS = None  # set e.g. 30000 if you want to cap DP context

# ---- Base folder in your Drive (edit once) ----
BASE = Path("/content")

# Use your three book.txt + QA CSV files here
JOBS = [
    {
        "name": "news",
        "book_txt": BASE / "news_book.txt",
        "qa_csv":   BASE / "news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv",
    },
    {
        "name": "scifi",
        "book_txt": BASE / "scifi_book.txt",
        "qa_csv":   BASE / "scifi_20260312T170655Z_df_qa.combined_story_minerva_balanced_150.csv",
    },
    {
        "name": "thriller",
        "book_txt": BASE / "book.txt",
        "qa_csv":   BASE / "thriller_20260311T152054Z_df_qa.combined_story_minerva_balanced_150.csv",
    },
]

OUT_DIR = BASE / "outputs_qwen25_14b_dp_rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
def ts():
    return datetime.now(UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def log_line(msg, log_path=None):
    line = f"[{ts()}] {msg}"
    print(line, flush=True)
    if log_path is not None:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")

def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def row_key(row: pd.Series, i: int):
    if "combined_q_idx" in row and pd.notna(row["combined_q_idx"]):
        return f"combined_q_idx:{int(row['combined_q_idx'])}"
    return f"row_idx:{int(i)}"

def split_book_into_chunks(book_text: str):
    chunks = []
    chapters = book_text.split("Chapter ")
    if len(chapters) > 1:
        for ci, ch in enumerate(chapters[1:], start=1):
            paras = [p.strip() for p in ch.split("\n\n") if p.strip()]
            for pi, p in enumerate(paras, start=1):
                chunks.append({
                    "chunk_id": len(chunks),
                    "label": f"Chapter {ci}, Paragraph {pi}",
                    "text": p
                })
    else:
        paras = [p.strip() for p in book_text.split("\n\n") if p.strip()]
        for pi, p in enumerate(paras, start=1):
            chunks.append({
                "chunk_id": len(chunks),
                "label": f"Paragraph {pi}",
                "text": p
            })
    return chunks

def build_rag_index(book_text: str, embedder: SentenceTransformer):
    chunks = split_book_into_chunks(book_text)
    X = embedder.encode([c["text"] for c in chunks], convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    idx = faiss.IndexFlatL2(X.shape[1])
    idx.add(X)
    return idx, chunks

def retrieve_chunks(question: str, idx, chunks, embedder, top_k: int):
    qv = embedder.encode([question], convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    D, I = idx.search(qv, top_k)
    out = []
    for rank, (d, j) in enumerate(zip(D[0], I[0]), start=1):
        c = chunks[int(j)]
        out.append({
            "rank": rank,
            "distance": float(d),
            "chunk_id": c["chunk_id"],
            "label": c["label"],
            "text": c["text"],
        })
    return out

def build_rag_prompt(question: str, retrieved):
    context = "\n\n".join([f"[{r['label']}]\n{r['text']}" for r in retrieved])
    return f"""# Episodic Memory Benchmark
You are participating in an episodic memory test based on retrieved text.
Answer using only the retrieved chunks. If unsure, say unsure.

## Retrieved Relevant Chunks:
{context}

## Question:
{question}

Return only the final answer.
"""

def build_dp_prompt(question: str, book_text: str, max_chars=None):
    src = book_text if (max_chars is None or max_chars <= 0) else book_text[:max_chars]
    return f"""# Episodic Memory Benchmark
You are participating in an episodic memory test.
Answer using only the book text. If unsure, say unsure.

## Book Text:
{src}

## Question:
{question}

Return only the final answer.
"""
@torch.inference_mode()
def generate_answer(model, tokenizer, user_prompt, max_new_tokens=80, temperature=0.0):
    messages = [
        {"role": "system", "content": "You are an expert in memory tests."},
        {"role": "user", "content": user_prompt},
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,   # important
    )

    if torch.cuda.is_available():
        model_inputs = {k: v.to("cuda") for k, v in model_inputs.items()}

    input_len = model_inputs["input_ids"].shape[-1]

    out = model.generate(
        **model_inputs,      # important (not model.generate(input_ids))
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=max(temperature, 1e-5),
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen = out[0, input_len:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip(), int(input_len)


In [4]:
!pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.4 MB/s eta 0:00:00


In [5]:
!pip uninstall -y transformers tokenizers
!pip install -U "transformers>=4.51.0" "tokenizers>=0.21.0" "accelerate>=0.34.0"

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.34.2
    Uninstalling accelerate-0.34.2:
      Successfully uninstalled accelerate-0.34.2
ERROR: pip's dependency resolver does not currently take into acco

In [6]:
import transformers
print(transformers.__version__)

4.46.3


In [1]:
!pip uninstall -y bitsandbytes
!pip install -U "bitsandbytes>=0.46.1"

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [6]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model.eval()

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
print("Model + embedder loaded.")

CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model + embedder loaded.


In [7]:
def run_job(job, mode="rag", limit=None):
    assert mode in {"rag", "dp"}

    name = job["name"]
    book_txt = Path(job["book_txt"])
    qa_csv = Path(job["qa_csv"])

    assert book_txt.exists(), f"Missing: {book_txt}"
    assert qa_csv.exists(), f"Missing: {qa_csv}"

    out_jsonl = OUT_DIR / f"{name}_qwen3_4b_{mode}.answers.jsonl"
    run_log = OUT_DIR / f"{name}_qwen3_4b_{mode}.run.log"

    log_line(f"run_start mode={mode} name={name}", run_log)
    log_line(f"book_txt={book_txt}", run_log)
    log_line(f"qa_csv={qa_csv}", run_log)
    log_line(f"output={out_jsonl}", run_log)

    book_text = book_txt.read_text(encoding="utf-8", errors="replace")
    df = pd.read_csv(qa_csv)
    if limit is not None:
        df = df.head(limit).copy()

    done = {r.get("row_key") for r in read_jsonl(out_jsonl) if r.get("status") == "ok"}
    log_line(f"resume_done={len(done)} rows_total={len(df)}", run_log)

    idx, chunks = (None, None)
    if mode == "rag":
        log_line(f"building_rag_index embed_model={EMBED_MODEL}", run_log)
        idx, chunks = build_rag_index(book_text, embedder)
        log_line(f"chunks_indexed={len(chunks)}", run_log)

    for i in tqdm(range(len(df))):
        row = df.iloc[i]
        rk = row_key(row, i)
        if rk in done:
            continue

        q = str(row["question"])
        q_idx = int(row["q_idx"]) if ("q_idx" in row and pd.notna(row["q_idx"])) else int(i)
        combined_q_idx = int(row["combined_q_idx"]) if ("combined_q_idx" in row and pd.notna(row["combined_q_idx"])) else None

        retrieved = None
        if mode == "rag":
            retrieved = retrieve_chunks(q, idx, chunks, embedder, TOP_K_RAG)
            prompt = build_rag_prompt(q, retrieved)
        else:
            prompt = build_dp_prompt(q, book_text, max_chars=DP_MAX_BOOK_CHARS)

        status = "ok"
        err = None
        answer = ""
        prompt_tokens = None
        try:
            answer, prompt_tokens = generate_answer(
                model=model,
                tokenizer=tokenizer,
                user_prompt=prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
            )
        except Exception as e:
            import traceback
            status = "error"
            err = f"{type(e).__name__}: {e}\n{traceback.format_exc(limit=5)}"


        rec = {
            "timestamp_utc": ts(),
            "status": status,
            "error": err,
            "mode": mode,
            "model_id": MODEL_ID,
            "row_key": rk,
            "row_idx": int(i),
            "q_idx": q_idx,
            "combined_q_idx": combined_q_idx,
            "question": q,
            "retrieval_type": str(row["retrieval_type"]) if "retrieval_type" in row and pd.notna(row["retrieval_type"]) else None,
            "get": str(row["get"]) if "get" in row and pd.notna(row["get"]) else None,
            "correct_answer": str(row["correct_answer"]) if "correct_answer" in row and pd.notna(row["correct_answer"]) else None,
            "prompt_tokens": prompt_tokens,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "answer": answer,
            "top_k": TOP_K_RAG if mode == "rag" else None,
            "retrieved_chunks": retrieved if mode == "rag" else None,
        }
        append_jsonl(out_jsonl, rec)

        if status == "ok":
            done.add(rk)

        if SLEEP_SEC > 0:
            time.sleep(SLEEP_SEC)

    log_line(f"run_done mode={mode} name={name} done_ok={len(done)}", run_log)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
for job in JOBS:
    run_job(job, mode="rag", limit=None)

[2026-04-02T16:43:37Z] run_start mode=rag name=news
[2026-04-02T16:43:37Z] book_txt=/content/news_book.txt
[2026-04-02T16:43:37Z] qa_csv=/content/news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T16:43:37Z] output=/content/outputs_qwen25_14b_dp_rag/news_qwen3_4b_rag.answers.jsonl
[2026-04-02T16:43:37Z] resume_done=1 rows_total=150
[2026-04-02T16:43:37Z] building_rag_index embed_model=sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[2026-04-02T16:43:44Z] chunks_indexed=405


  0%|          | 0/150 [00:00<?, ?it/s]

[2026-04-02T16:52:12Z] run_done mode=rag name=news done_ok=150
[2026-04-02T16:52:12Z] run_start mode=rag name=scifi
[2026-04-02T16:52:12Z] book_txt=/content/scifi_book.txt
[2026-04-02T16:52:12Z] qa_csv=/content/scifi_20260312T170655Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T16:52:12Z] output=/content/outputs_qwen25_14b_dp_rag/scifi_qwen3_4b_rag.answers.jsonl
[2026-04-02T16:52:12Z] resume_done=0 rows_total=150
[2026-04-02T16:52:12Z] building_rag_index embed_model=sentence-transformers/all-MiniLM-L6-v2


...

[2026-04-02T17:02:34Z] run_done mode=rag name=scifi done_ok=150
[2026-04-02T17:02:34Z] run_start mode=rag name=thriller
[2026-04-02T17:02:34Z] book_txt=/content/book.txt
[2026-04-02T17:02:34Z] qa_csv=/content/thriller_20260311T152054Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:02:34Z] output=/content/outputs_qwen25_14b_dp_rag/thriller_qwen3_4b_rag.answers.jsonl
[2026-04-02T17:02:34Z] resume_done=0 rows_total=150
[2026-04-02T17:02:34Z] building_rag_index embed_model=sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

[2026-04-02T17:02:54Z] chunks_indexed=825


  0%|          | 0/150 [00:00<?, ?it/s]

[2026-04-02T17:11:58Z] run_done mode=rag name=thriller done_ok=150


In [11]:
for job in JOBS:
    run_job(job, mode="dp", limit=None)

[2026-04-02T17:14:29Z] run_start mode=dp name=news
[2026-04-02T17:14:29Z] book_txt=/content/news_book.txt
[2026-04-02T17:14:29Z] qa_csv=/content/news_news_llm_retry_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:14:29Z] output=/content/outputs_qwen25_14b_dp_rag/news_qwen3_4b_dp.answers.jsonl
[2026-04-02T17:14:29Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

[2026-04-02T17:14:53Z] run_done mode=dp name=news done_ok=0
[2026-04-02T17:14:53Z] run_start mode=dp name=scifi
[2026-04-02T17:14:53Z] book_txt=/content/scifi_book.txt
[2026-04-02T17:14:53Z] qa_csv=/content/scifi_20260312T170655Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:14:53Z] output=/content/outputs_qwen25_14b_dp_rag/scifi_qwen3_4b_dp.answers.jsonl
[2026-04-02T17:14:53Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

[2026-04-02T17:15:23Z] run_done mode=dp name=scifi done_ok=0
[2026-04-02T17:15:23Z] run_start mode=dp name=thriller
[2026-04-02T17:15:23Z] book_txt=/content/book.txt
[2026-04-02T17:15:23Z] qa_csv=/content/thriller_20260311T152054Z_df_qa.combined_story_minerva_balanced_150.csv
[2026-04-02T17:15:23Z] output=/content/outputs_qwen25_14b_dp_rag/thriller_qwen3_4b_dp.answers.jsonl
[2026-04-02T17:15:23Z] resume_done=0 rows_total=150


  0%|          | 0/150 [00:00<?, ?it/s]

[2026-04-02T17:16:33Z] run_done mode=dp name=thriller done_ok=0
